# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import os
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
logging.info(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
logging.info(f"✅ Torch CUDA available: {cuda_test}")
device_name = torch.cuda.get_device_name(0)
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"🖥️ Device Name: {device_name} | Device reference: {torch_device.type}")

### machine learning (scikit-learn)
import math
import pprint
import numpy as np
import pandas as pd
import torch
from sklearn.pipeline import Pipeline
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import warnings
warnings.filterwarnings(
    "ignore",
    message="mtime may not be reliable on this filesystem, falling back to numerical ordering"
)
from transformers import (
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed  # type: ignore
)
from tsfm_public import (
    TinyTimeMixerForPrediction,
    TimeSeriesForecastingPipeline,
    TrackingCallback,
    count_parameters,
)
from tsfm_public.toolkit.time_series_preprocessor import get_datasets, prepare_data_splits
from tsfm_public.toolkit.visualization import plot_predictions

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.preprocessing_project_specific as pps
import smartcheck.deep_learning_project_specific as dlps

# 2. Loading and Preprocessing

## 2.1 Loading data

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Data Refactoring pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    # "date_et_heure_de_comptage",
    "date_et_heure_de_comptage_local",
    # "date_et_heure_de_comptage_utc",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("convert_datetime", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage",
                                                                   for_sarimax=True)),
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

# 3. Modeling Training and Prediction

## 3.1 Contextual variables

#### For the model

In [ ]:
timestamp_column = "date_et_heure_de_comptage_local"
target_columns = ["comptage_horaire"]
context_length = 512
fcm_context_length = 48 # 2 jours
prediction_length = 96
# Output directory for writing evaluation results.
OUT_DIR = "ttm_results.model"
# Return this percent of the original dataset when getting train/test splits.
fewshot_fraction = 1
# Important training parameters
learning_rate: float = 0.0002
num_epochs: int = 200
patience: int = 10
batch_size: int = 32
steps_per_epoch=math.ceil(10000 / batch_size)
set_seed(42)
split_config = {"train": 0.6, "test": 0.2}
column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": [],
    "target_columns": target_columns,
    "categorical_columns": [
        "vacances_scolaires",
        "weather_code_wmo_code_category",
    ],
    "control_columns": [
        "jour_ferie",
        "vacances_scolaires",
    ],
    "observable_columns": [
        "temperature_2m_c",
        "rain_mm",
        "snowfall_cm",
        "weather_code_wmo_code_category",
    ],
}

#### For the experiments

In [ ]:
dict_compteurs = {
    # "experiment_1": {
    #     "key": ('135 avenue Daumesnil','SE-NO'),
    #     "name": "Daumesnil_S-N",
    #     "sub_range": (0,),
    #     # "last_checkpoint": "checkpoint-1",
    #     # "best_checkpoint": "checkpoint-332",
    # },
    "experiment_2": {
        "key": ('102 boulevard de Magenta', 'SE-NO'),
        "name": "Magenta-O-E",
        "sub_range": (0,),
        # "last_checkpoint": "checkpoint-1",
        "best_checkpoint": "checkpoint-340",
    },
    # "experiment_3": {
    #     "key": ('Totem 73 boulevard de Sébastopol', 'S-N'),
    #     "name": "Sébastopol_S-N",
    #     "sub_range": (0,),
    #     # "last_checkpoint": "checkpoint-1",
    #     # "best_checkpoint": "checkpoint-248",
    # },
}
# Enrich the experiment checkpoint dir and display the dictionnary
for experiment, params in dict_compteurs.items():
    name = params["name"]
    if "last_checkpoint" in params:
        last_checkpoint_dir = os.path.join(OUT_DIR, f"output_{name}", params["last_checkpoint"])
        dict_compteurs[experiment]["last_checkpoint_dir"] = last_checkpoint_dir
    if "best_checkpoint" in params:
        best_checkpoint_dir = os.path.join(OUT_DIR, f"output_{name}", params["best_checkpoint"])
        dict_compteurs[experiment]["best_checkpoint_dir"] = best_checkpoint_dir
logging.info("\n" + pprint.pformat(dict_compteurs))

## 3.2 Data Viz of Time Series per counter

In [ ]:
# splitted dataframe per counter
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for experiment, exp_params in dict_compteurs.items():
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {exp_params} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue

    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    fig, axs = plt.subplots(len(target_columns), 1, figsize=(10, 2 * len(target_columns)), squeeze=False)
    for ax, target_column in zip(axs, target_columns):
        ax[0].plot(df_compteur_sub[timestamp_column], df_compteur_sub[target_column])
    plt.show()

## 3.3 Transfert learning with preprocessing and training

#### Définition et ajustement du modèle granite pour finetuning avec gel des couches pré-entrainées
> Environ 500k paramètres gelés mais il reste 1,2M paramètres ajoutés via notre contexte de variables exogènes

In [ ]:
def fine_tune_model(output_dir, logging_dir):
    # Define preprocessor
    finetune_timeseries_preprocessor = dlps.SafeTimeSeriesPreprocessorOrdinal(
        **column_specifiers,
        context_length=context_length,
        prediction_length=prediction_length,
        scaling=True,
        freq="h",
        encode_categorical=True,
        scale_categorical_columns=True,
        scaler_type="standard",  # type: ignore
    )

    # Define model from Hugging Face
    finetune_forecast_model = TinyTimeMixerForPrediction.from_pretrained(
        "ibm-granite/granite-timeseries-ttm-r2",  # Name of the model on HuggingFace.
        num_input_channels=finetune_timeseries_preprocessor.num_input_channels,
        prediction_channel_indices=finetune_timeseries_preprocessor.prediction_channel_indices,
        exogenous_channel_indices=finetune_timeseries_preprocessor.exogenous_channel_indices,
        fcm_use_mixer=True,
        fcm_context_length=fcm_context_length,  
        enable_forecast_channel_mixing=True,
        decoder_mode="mix_channel",
    )

    # Freeze the backbone of the model
    logging.info(f"Number of params before freezing backbone {count_parameters(finetune_forecast_model)}")
    for param in finetune_forecast_model.backbone.parameters():
        param.requires_grad = False
    logging.info(f"Number of params after freezing the backbone {count_parameters(finetune_forecast_model)}")

    # Set the training arguments
    logging.info(f"Learning Rate = {learning_rate} | {num_epochs} epoch(s) |"
                 f" utilisation du {'GPU' if (torch_device.type == 'cuda') else 'CPU'}")
    finetune_forecast_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        learning_rate=learning_rate,
        num_train_epochs=num_epochs,
        do_eval=True,
        eval_strategy="epoch",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        dataloader_pin_memory=True,
        report_to=None,
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=1,
        logging_dir=logging_dir,  # Make sure to specify a logging directory
        load_best_model_at_end=True,  # Load the best model when training ends
        metric_for_best_model="eval_loss",  # Metric to monitor for early stopping
        greater_is_better=False,  # For loss
        use_cpu=torch_device.type != "cuda",
    )

    # Create the early stopping callback
    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=patience,  # Number of epochs with no improvement after which to stop
        early_stopping_threshold=0.00001,  # Minimum improvement required to consider as improvement
    )
    tracking_callback = TrackingCallback()

    # Define an optimizer and scheduler
    finetune_optimizer = AdamW(finetune_forecast_model.parameters(), lr=learning_rate)
    finetune_scheduler = OneCycleLR(
        finetune_optimizer,
        learning_rate,
        epochs=num_epochs,
        steps_per_epoch=steps_per_epoch,
    )
    return (
        finetune_timeseries_preprocessor,
        finetune_forecast_model,
        finetune_forecast_args,
        finetune_optimizer, finetune_scheduler,
        early_stopping_callback,
        tracking_callback
    )

#### Boucle d'entrainement du modèle

In [ ]:
# splitted dataframe per counter
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

model_results = {}
# Make a forecast on the target column given the input data.
for experiment, exp_params in dict_compteurs.items():
    # Filtrage de l'experience et de son dataset associé
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {exp_params} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue
    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    # Definition et fine tuning du modèle
    name = exp_params["name"]
    preproc_dir = os.path.join(OUT_DIR, f"preproc_{name}")
    output_dir = os.path.join(OUT_DIR, f"output_{name}")
    logging_dir = os.path.join(OUT_DIR, f"log_{name}")
    (
        tsp,
        model,
        args,
        optimizer,
        scheduler,
        early_stop_cb,
        tracking_cb
    ) = fine_tune_model(output_dir, logging_dir)

    # Split train, valid, test for dataframes and datasets for training
    train_df, valid_df, test_df = prepare_data_splits(  # type: ignore
        df_compteur_sub,
        context_length=context_length,
        split_config=split_config  # type: ignore
    )
    logging.info(f"Dataframe lengths: train = {len(train_df)}, val = {len(valid_df)}, test = {len(test_df)}")
    train_dataset, valid_dataset, test_dataset = get_datasets(  # type: ignore
        tsp,
        df_compteur_sub,
        split_config,  # type: ignore
        stride=prediction_length,
        fewshot_fraction=fewshot_fraction,
        fewshot_location="first",
        use_frequency_token=model.config.resolution_prefix_tuning,
    )
    logging.info(f"Dataset batch lengths: train = {len(train_dataset)}, val = {len(valid_dataset)}, test = {len(test_dataset)}")

    # Définition du modèle et de son trainer
    finetune_forecast_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        callbacks=[early_stop_cb, tracking_cb],
        optimizers=(optimizer, scheduler)  # type: ignore
    )

    # Priorité au reentrainement à partir du best checkpoint
    best_checkpoint = dlps.train_or_resume(finetune_forecast_trainer, exp_params)

    # sauvegarde des résultats en mémoire
    model_results[experiment] = {
        "exp_params": exp_params,
        "train_df": train_df,
        "valid_df": valid_df,
        "test_df": test_df,
        "best_checkpoint": best_checkpoint,
        "output_dir": output_dir,
        "logging_dir": logging_dir,
        "preproc_dir": preproc_dir,
        "model": model,
        "tsp": tsp,
    }

    # sauvegarde sur disque de l'experience via joblib / yaml (preprocessor inclus)
    dlps.save_granite_model(experiment, model_results[experiment])
    dlps.save_preprocessor_state(tsp, preproc_dir)
    

## 3.3 Predictions

#### From memory context

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_train = model_result["train_df"]
    df_valid = model_result["valid_df"]
    df_test = model_result["test_df"]
    model = model_result["model"]
    tsp = model_result["tsp"]
    logging.info(f"\n\nParamètres {exp_params} :\n")
    checkpoint_dir = os.path.join(model_result['output_dir'], model_result['best_checkpoint'])
    preproc_dir = os.path.join(model_result['preproc_dir'])
    
    # Create the evaluation pipeline
    pipeline = TimeSeriesForecastingPipeline(
        model=model,
        device=torch_device,
        feature_extractor=tsp,
        batch_size=batch_size,
    )

    # Collect the test df and run the predictions
    predictions_df_test = pipeline(df_test)  # type: ignore

    # Print/Plot the predictions
    plot_predictions(
        input_df=df_test,
        predictions_df=predictions_df_test,  # type: ignore
        freq="h",
        timestamp_column=timestamp_column,
        channel=target_column,
        # we check the prediction on each of the 3 previous week and in the future
        indices=[ -24*7*3, -24*7*2, -24*7*1, -1],
        num_plots=4,
    )
    plt.show()

#### From Disk context

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_train = model_result["train_df"]
    df_valid = model_result["valid_df"]
    df_test = model_result["test_df"]
    logging.info(f"\n\nParamètres {exp_params} :\n")
    checkpoint_dir = os.path.join(model_result['output_dir'], model_result['best_checkpoint'])
    preproc_dir = os.path.join(model_result['preproc_dir'])
    
    # Reload from disk for safe check
    reloaded_tsp = dlps.load_preprocessor_state(preproc_dir)
    reloaded_model = dlps.load_model_from_checkpoint(TinyTimeMixerForPrediction, checkpoint_dir, torch_device.type)

    # Create the evaluation pipeline
    pipeline = TimeSeriesForecastingPipeline(
        model=reloaded_model,
        device=torch_device,
        feature_extractor=reloaded_tsp,
        batch_size=batch_size,
    )

    # Collect the test df and run the predictions
    predictions_df_test = pipeline(df_test)  # type: ignore

    # Print/Plot the predictions
    plot_predictions(
        input_df=df_test,
        predictions_df=predictions_df_test,  # type: ignore
        freq="h",
        timestamp_column=timestamp_column,
        channel=target_column,
        # we check the prediction on each of the 3 previous week and in the future
        indices=[ -24*7*3, -24*7*2, -24*7*1, -1],
        num_plots=4,
    )
    plt.show()